## Whisper STT Implementation Lab

In [1]:
!pip install openai pydub   

In [2]:
"""
Whisper STT Implementation Lab
Author: Carlos Felipe Valencia
Description: Transcribe meeting audio using OpenAI Whisper API, including chunking, timestamps, and prompted/unprompted transcription.
"""

import os
import json
import time
import subprocess
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Create project directories
AUDIO_DIR = Path("audio")
CHUNKS_DIR = Path("chunks")
OUTPUT_DIR = Path("transcriptions")

AUDIO_DIR.mkdir(exist_ok=True)
CHUNKS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("=" * 50)
print("ENVIRONMENT SETUP")
print("=" * 50)
print("OpenAI client initialized successfully!")
print("API key loaded:", os.getenv("OPENAI_API_KEY") is not None)
print(f"Audio directory: {AUDIO_DIR}")
print(f"Chunks directory: {CHUNKS_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print("FFmpeg will be used directly for audio processing.")

ENVIRONMENT SETUP
OpenAI client initialized successfully!
API key loaded: True
Audio directory: audio
Chunks directory: chunks
Output directory: transcriptions
FFmpeg will be used directly for audio processing.


In [3]:
result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)

if result.returncode == 0:
    print("FFmpeg is installed and working!")
else:
    print("FFmpeg not found.")

FFmpeg is installed and working!


In [4]:
audio_path = AUDIO_DIR / "MI072clip.mp3"

if audio_path.exists():
    print("✅ Audio encontrado:", audio_path)
else:
    print("❌ No se encontró el audio")

✅ Audio encontrado: audio\MI072clip.mp3


In [5]:
def transcribe_audio(file_path):
    with open(file_path, "rb") as audio_file:
        result = client.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe",
            file=audio_file
        )
    return result

result = transcribe_audio(audio_path)

print("\nTRANSCRIPTION:\n")
print(result.text)


TRANSCRIPTION:

They didn't accept you if you weren't one of them. Several of them used to tell me that I shouldn't tell people that I was colored because I didn't look it. And I still didn't feel this until I was older and started trying to get a job. How did it manifest itself then? The first time was when I went to the telephone company and I was very young and I wanted to apply for a job. And the woman there that in personnel asked me had I had any experience? And I told her no, that I had never held a job and that I had no kind of experience at all. And she did say that, well, she felt she had a job that I would enjoy, that I could learn very well. And she started filling out an application. And she was filling it out until she came to nationality, and she asked me what was my nationality. And I was stunned because I had always felt that anyone could look at me and see what I was. And I told her I was colored. And she turned as red as a beet and she said she was sorry, she didn't

## Step 4: Transcription with Prompts.


In [6]:
def transcribe_audio_with_prompt(file_path, prompt):
    """
    Transcribe audio using a prompt to guide Whisper.
    """
    with open(file_path, "rb") as audio_file:
        result = client.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe",
            file=audio_file,
            prompt=prompt
        )
    return result


meeting_prompt = """
This is a business meeting. The speakers may mention project planning,
AI tools, automation, APIs, transcription, clients, deadlines, and business strategy.
Please preserve technical terms, names, and business vocabulary as accurately as possible.
"""

guided_result = transcribe_audio_with_prompt(audio_path, meeting_prompt)

print("\nGUIDED TRANSCRIPTION:\n")
print(guided_result.text)


GUIDED TRANSCRIPTION:

They didn't accept you if you weren't one of them. Several of them used to tell me that I shouldn't tell people that I was colored because I didn't look it. And I still didn't feel this until I was older and started trying to get a job. How did it manifest itself then? The first time was when I went to the telephone company and I was very young, and I wanted to apply for a job. And the woman there that in personnel asked me had I had any experience, and I told her no, that I had never held a job and that I had no kind of experience at all. And she did say that, well, she felt she had a job that I would enjoy, that I could learn very well. And she started filling out an application. And she was filling it out until she came to nationality, and she asked me what was my nationality. And I was stunned because I had always felt that anyone could look at me and see what I was. And I told her I was colored, and she turned as red as a beak, and she said she was sorry, s

## Step 5: Transcription Without Prompts (Unguided Approach)

In [7]:
print("=" * 50)
print("STEP 5: GUIDED VS UNGUIDED COMPARISON")
print("=" * 50)

# Unguided transcription: without prompt
unguided_result = transcribe_audio(audio_path)

# Guided transcription: with prompt
guided_result = transcribe_audio_with_prompt(audio_path, meeting_prompt)

print("\n--- UNGUIDED TRANSCRIPTION ---\n")
print(unguided_result.text)

print("\n--- GUIDED TRANSCRIPTION ---\n")
print(guided_result.text)

STEP 5: GUIDED VS UNGUIDED COMPARISON

--- UNGUIDED TRANSCRIPTION ---

They didn't accept you if you weren't one of them. Several of them used to tell me that I shouldn't tell people that I was colored because I didn't look it. And I still didn't feel this until I was older and started trying to get a job. How did it manifest itself then? The first time was when I went to the telephone company, and I was very young, and I wanted to apply for a job. And the woman there in personnel asked me had I had any experience, and I told her no, that I had never held a job and that I had no kind of experience at all. And she did say that, well, she felt she had a job that I would enjoy, that I could learn very well. And she started filling out an application. And she was filling it out until she came to nationality, and she asked me what was my nationality. And I was stunned because I had always felt that anyone could look at me and see what I was. And I told her I was colored. And she turned as r

## Step 5: Guided vs Unguided Transcription

In this step, I compared two transcription approaches:

- **Unguided transcription:** The audio was transcribed without additional context.
- **Guided transcription:** A prompt was added to provide context about the meeting topic and expected vocabulary.

The guided approach can improve accuracy when the audio contains technical terms, names, business vocabulary, or domain-specific language. The unguided approach is useful as a baseline, while the guided transcription may produce more accurate and context-aware results.

## Step 6: Implementing Audio Chunking


In [8]:
!python -m pip install sounddevice scipy

In [11]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wavfile
import io
import time
from IPython.display import Audio, display

# Record longer audio
duration = 15  # total seconds
sample_rate = 16000
chunk_duration = 5  # seconds per chunk

print(f"🎤 Recording for {duration} seconds... Speak continuously!")
audio = sd.rec(
    int(duration * sample_rate),
    samplerate=sample_rate,
    channels=1,
    dtype="float32"
)
sd.wait()
audio = audio.flatten()
print("✅ Recording complete!")

# Play full audio
display(Audio(audio, rate=sample_rate))

# Split audio into chunks
chunk_size = chunk_duration * sample_rate
chunks = []

for i in range(0, len(audio), chunk_size):
    chunk_audio = audio[i:i + chunk_size]
    start_offset = i / sample_rate

    chunks.append({
        "chunk_index": len(chunks) + 1,
        "audio": chunk_audio,
        "start_offset": start_offset
    })

print(f"\n🔪 Split into {len(chunks)} chunks")

for chunk in chunks:
    print(
        f"Chunk {chunk['chunk_index']} | "
        f"Start offset: {chunk['start_offset']:.2f}s | "
        f"Samples: {len(chunk['audio'])}"
    )

🎤 Recording for 15 seconds... Speak continuously!
✅ Recording complete!



🔪 Split into 3 chunks
Chunk 1 | Start offset: 0.00s | Samples: 80000
Chunk 2 | Start offset: 5.00s | Samples: 80000
Chunk 3 | Start offset: 10.00s | Samples: 80000


In [10]:
chunks = split_audio_ffmpeg(audio_path, chunk_length_seconds=60)

print("\nTotal chunks created:", len(chunks))
print(chunks[:3])

NameError: name 'split_audio_ffmpeg' is not defined

The audio was divided into smaller chunks to simulate processing long meeting recordings. Although the sample audio was relatively short, chunking demonstrates how larger recordings can be processed efficiently while preserving timestamps.

In [ ]:
def transcribe_chunk_with_timestamps(chunk):
    """
    Transcribe one audio chunk from file path and adjust timestamps.
    """

    chunk_path = chunk["chunk_path"]
    start_offset = chunk["start_offset"]

    with open(chunk_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            response_format="verbose_json",
            timestamp_granularities=["segment"]
        )

    adjusted_segments = []

    for segment in transcript.segments:
        adjusted_segments.append({
            "chunk_index": chunk["chunk_index"],
            "start": segment.start + start_offset,
            "end": segment.end + start_offset,
            "text": segment.text.strip()
        })

    return adjusted_segments

In [ ]:
print("\n" + "=" * 50)
print("TRANSCRIBING CHUNKS WITH TIMESTAMPS")
print("=" * 50)

all_segments = []

for chunk in chunks:
    print(f"Processing chunk {chunk['chunk_index']} of {len(chunks)}...")

    try:
        segments = transcribe_chunk_with_timestamps(chunk)
        all_segments.extend(segments)

        print(f"✅ Chunk {chunk['chunk_index']} transcribed successfully")
        time.sleep(1)

    except Exception as e:
        print(f"⚠️ Error transcribing chunk {chunk['chunk_index']}: {e}")

print("\n📝 Complete Transcription with Timestamps:")
print("-" * 50)

for segment in all_segments:
    print(f"[{segment['start']:.2f}s - {segment['end']:.2f}s] {segment['text']}")


TRANSCRIBING CHUNKS WITH TIMESTAMPS
Processing chunk 1 of 2...
✅ Chunk 1 transcribed successfully
Processing chunk 2 of 2...
✅ Chunk 2 transcribed successfully

📝 Complete Transcription with Timestamps:
--------------------------------------------------
[0.00s - 3.28s] They didn't accept you if you weren't one of them.
[3.28s - 7.52s] Several of them used to tell me that I shouldn't tell people that I was colored
[7.52s - 10.08s] because I didn't look it.
[10.08s - 21.16s] And I still didn't feel this until I was older and started trying to get a job.
[21.16s - 23.88s] How did it manifest itself then?
[24.00s - 32.00s] The first time was when I went to the telephone company, and I was very young, and I wanted
[32.00s - 34.72s] to apply for a job.
[34.72s - 41.92s] And the woman there in personnel asked me had I had any experience, and I told her no,
[41.92s - 47.96s] that I had never held a job, and that I had no kind of experience at all.
[47.96s - 52.80s] And she did say that, well,

## Step 7: Transcribing Chunks with Timestamps

In this step, each audio chunk was transcribed separately using Whisper. Since each chunk represents a different section of the original recording, I adjusted the timestamps by adding the chunk start offset.

This allowed the final transcript to preserve the correct timing from the full original audio. The result is a complete transcription with readable timestamps for each segment, making it easier to search, review, and reference specific parts of the audio.

In [ ]:
# Helper function to format seconds into SRT timestamp format
def format_srt_time(seconds):
    """
    Convert seconds to SRT timestamp format: HH:MM:SS,mmm
    """
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    milliseconds = int((seconds - int(seconds)) * 1000)

    return f"{hours:02}:{minutes:02}:{secs:02},{milliseconds:03}"

In [ ]:
txt_path = OUTPUT_DIR / "transcription_with_timestamps.txt"

with open(txt_path, "w", encoding="utf-8") as f:
    f.write("Transcription with Timestamps\n")
    f.write("=" * 50 + "\n\n")

    for segment in all_segments:
        f.write(
            f"[{segment['start']:.2f}s - {segment['end']:.2f}s] "
            f"{segment['text']}\n"
        )

print(f"✅ TXT exported to: {txt_path}")

✅ TXT exported to: transcriptions\transcription_with_timestamps.txt


In [ ]:
json_path = OUTPUT_DIR / "transcription_with_timestamps.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(all_segments, f, indent=2, ensure_ascii=False)

print(f"✅ JSON exported to: {json_path}")

✅ JSON exported to: transcriptions\transcription_with_timestamps.json


In [ ]:
srt_path = OUTPUT_DIR / "transcription_with_timestamps.srt"

with open(srt_path, "w", encoding="utf-8") as f:
    for index, segment in enumerate(all_segments, start=1):
        start_time = format_srt_time(segment["start"])
        end_time = format_srt_time(segment["end"])

        f.write(f"{index}\n")
        f.write(f"{start_time} --> {end_time}\n")
        f.write(f"{segment['text']}\n\n")

print(f"✅ SRT exported to: {srt_path}")

✅ SRT exported to: transcriptions\transcription_with_timestamps.srt


In [ ]:
print("\nExported files:")
print(txt_path)
print(json_path)
print(srt_path)

print("\nFirst segment preview:")
print(all_segments[0])


Exported files:
transcriptions\transcription_with_timestamps.txt
transcriptions\transcription_with_timestamps.json
transcriptions\transcription_with_timestamps.srt

First segment preview:
{'chunk_index': 1, 'start': 0.0, 'end': 3.2799999713897705, 'text': "They didn't accept you if you weren't one of them."}


## Step 8: Exporting with Timestamps

In this step, I exported the final timestamped transcription into multiple formats.

The TXT file provides a human-readable transcript with timestamps. The JSON file stores the transcription as structured data, which is useful for search, analysis, and integration with other systems. The SRT file follows a subtitle format, making it suitable for video captions or media players.

Exporting the transcription in multiple formats increases usability and makes the results adaptable for different business and technical needs.